# Ü6.1 — Verschachteltes JSON relationalisieren

`raw.customers` einlesen (geschachtelt: `address`, `contacts[]`), den
mischtypigen `loyalty_points` mit `resolveChoice` vereinheitlichen, dann mit
`Relationalize` in flache Tabellen zerlegen (Root + Array-Child) und beide
als Parquet schreiben.

**Voraussetzung:** Crawler hat `raw.customers` katalogisiert.
`loyalty_points` ist absichtlich mischtypig (`1200` vs `"gold"`) → ohne
`resolveChoice` bekommt man einen `choice`-Typ.

Die Code-Zellen sind schon vorgefertigt — du ersetzt nur die `____`-Platzhalter.
Die Syntax steht; du entscheidest nur, **was wohin** gehört. Über jeder Zelle
findest du je eine Leitfrage pro `____`.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping, Filter
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Lesen — geschachteltes JSON

In [ ]:
customers = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="customers", transformation_ctx="customers"
)
# Der choice-Typ auf loyalty_points ist hier gut sichtbar:
customers.printSchema()

## 2) resolveChoice — loyalty_points vereinheitlichen

Die Zelle unten ist vorgefertigt — ersetze nur die `____`. Frage: Welche Spalte
ist mischtypig, und mit welcher Strategie führst du Zahl und Text verlustfrei
zusammen?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____feld   : Welche Spalte ist mischtypig (Zahl vs. Text) und muss vereinheitlicht werden?
#   ____aktion : Wie führst du beide Ausprägungen verlustfrei zusammen?
#                (Muster: "cast:<typ>", "make_struct", "project:<typ>")
resolved = customers.resolveChoice(specs=[("____feld", "____aktion")])
resolved.printSchema()

## 3) Relationalize — in flache Frames zerlegen

`Relationalize` gibt dir eine Sammlung flacher Frames zurück: den Root-Frame
plus je einen Child-Frame pro Array-Feld (Muster: `<root>_<feld>`). Ersetze
unten die `____` — welchen Frame zerlegst du, wie heißt der Root, und wohin
dürfen die Zwischendaten?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____frame   : Welchen Frame zerlegst du — den rohen oder den bereits vereinheitlichten?
#   ____root    : Wie soll der Root-Frame heißen? (dieser Name wird Präfix der Child-Frames: <root>_<feld>)
#   ____staging : Wohin dürfen die Zwischendaten? (S3-Pfad, Muster: "s3://<bucket>/temp/relationalize/")
from awsglue.transforms import Relationalize

frames = Relationalize.apply(frame=____frame, name="____root", staging_path=____staging)
print("erzeugte Frames:", frames.keys())

## 4) Root- und Child-Frame herausgreifen

Schau dir oben `frames.keys()` an: dort stehen die Namen der erzeugten Frames.
Ersetze unten die `____` durch die passenden Schlüssel — welcher ist der Root,
welcher der Child-Frame (Muster: `<root>_<feld>`)?

In [ ]:
# Fülle die ____ aus. Die Frame-Namen stehen in frames.keys() (Zelle oben).
#   ____rootname  : Welcher Schlüssel ist der Root-Frame? (= der name aus Schritt 3)
#   ____childname : Welcher Schlüssel ist der Child-Frame? (Muster: <root>_<feld>)
root = frames.select("____rootname")
child = frames.select("____childname")
root.printSchema()
child.printSchema()

## 5) Beide als Parquet schreiben

Zum Schluss beide Frames nach `processed/` schreiben. Ersetze die `____` —
in welche Unterordner kommen Root und Child, und welcher Schreibmodus
überschreibt vorhandene Daten?

In [ ]:
# Fülle die ____ aus. Beide Frames als Parquet nach processed/ schreiben.
#   ____modus     : Welcher Schreibmodus überschreibt vorhandene Daten?
#   ____rootpfad  : In welchen Unterordner kommt der Root-Frame?
#   ____childpfad : In welchen Unterordner kommt der Child-Frame?
OUT = "s3://<bucket>/processed/"
root.toDF().write.mode("____modus").parquet(OUT + "____rootpfad/")
child.toDF().write.mode("____modus").parquet(OUT + "____childpfad/")
print("root rows:", root.count(), "| child rows:", child.count())